# Parallel Chain
Let's use the LCEL language to build a parallel chain that returns the answer, a summary and the sentiment of the answer.


In [ ]:
%pip install -qU langchain-ollama --quiet

Import libraries and Nvidia key

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

## Main model

In [3]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant"),
    ("user", "{input}")
    ])

## Summary Chain

In [40]:
summary_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant that makes great summaries, Given the text, generate a concise summary."),
    ("user", "{input}")
    ])

In [43]:
summary_chain = summary_prompt | llm | StrOutputParser()

Test the summary chain in isolation

In [44]:
response = summary_chain.invoke({"input": "I am so happy today. The sky is blue and I am on holidays. I might go for a walk in the park."})
print(response)

The speaker is happy on a blue-sky holiday and plans to possibly take a walk in the park.


## Sentiment chain

In [28]:
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant that specializes on sentiment analysis. \
     Given a text, classify it as positive, neutral or negative. Return a single word with your classification. \
     No formatting or special characters."),
    ("user", "{input}")
    ])

In [46]:
sentiment_chain = sentiment_prompt | llm | StrOutputParser()

Test the summary chain

In [47]:
response = sentiment_chain.invoke({"input": "I am so happy today. The sky is blue and I am on holidays. I might go for a walk in the park."})
print(response)

Positive


## Parallel chain

In [ ]:
#    answer=RunnablePassthrough(),
parallel_chain = RunnableParallel (
    answer=RunnableLambda(lambda x: x["input"]),
    summary=summary_chain,
    sentiment=sentiment_chain,
)

In [89]:
response = parallel_chain.invoke({"input": "I am so happy today. The sky is blue and I am on holidays. I might go for a walk in the park."})

In [90]:
import pprint
pprint.pprint(response)

{'answer': 'I am so happy today. The sky is blue and I am on holidays. I might '
           'go for a walk in the park.',
 'sentiment': 'Positive',
 'summary': 'The user is feeling happy on a blue-sky holiday and plans to '
            'possibly take a walk in the park.'}


## Main chain
Concatenate the LLM with the parallel chain

In [94]:
#preprocessor = RunnableLambda(lower_case) | RunnableLambda(mask_credit_card_numbers)
main_chain = prompt | llm | {"input": StrOutputParser()} | parallel_chain

Testing the main chain

In [ ]:
# This is a positive one
response = main_chain.invoke({"input": "I am so happy today. By the way what is the colour of the sky?"})

In [96]:
import pprint
pprint.pprint(response)

{'answer': "I'm so happy to hear that you're having a wonderful day! 😊\n"
           '\n'
           'As for the color of the sky, the answer is actually **it depends '
           'on the time of day and the weather!**\n'
           '\n'
           '*   **On a clear, sunny day:** It is usually a beautiful, bright '
           '**blue**. (This is because of a phenomenon called Rayleigh '
           'scattering, where the atmosphere scatters the shorter blue '
           'wavelengths of light more effectively than other colors.)\n'
           '*   **At sunrise or sunset:** It often turns vibrant shades of '
           '**orange, pink, red, and yellow**.\n'
           '*   **On a cloudy or stormy day:** It can look **gray** or even '
           'dark.\n'
           '\n'
           'So, while the most common answer is **blue**, the sky is always '
           'putting on a beautiful, ever-changing show! 🎨',
 'sentiment': 'Positive',
 'summary': 'The color of the sky is variable and depends 

In [ ]:
# So hard to get negative thoughts from the LLM :D
response = main_chain.invoke({"input": "I feel kind of sad. Can you give me 3 negative thoughts. Please, don't try to be conforting"})

In [102]:
pprint.pprint(response)

{'answer': '1. All your efforts will eventually fade into irrelevance.\n'
           '2. Nothing you do today will fundamentally change the trajectory '
           'of your life.\n'
           '3. You are inherently flawed in ways you will never be able to '
           'fix.',
 'sentiment': 'Negative',
 'summary': 'This passage conveys a deeply fatalistic and pessimistic outlook, '
            'suggesting that human effort is ultimately futile, individual '
            "actions lack the power to fundamentally change one's life path, "
            'and inherent flaws are permanent.'}
